In [1]:
from scapy.all import rdpcap
import pandas as pd

pcap_file = "2018/03/20/eth1-20180320.0415.1521537300"

packets = rdpcap(pcap_file)

rows = []

for pkt in packets:
    if "IP" in pkt:
        rows.append({
            "time": float(pkt.time),
            "src_ip": pkt["IP"].src,
            "dst_ip": pkt["IP"].dst,
            "protocol": pkt["IP"].proto,
            "length": len(pkt),
        })

df = pd.DataFrame(rows)
print(df.head())
print(df.shape)

           time        src_ip         dst_ip  protocol  length
0  1.521519e+09  192.168.0.29  34.209.123.19         6    1514
1  1.521519e+09  192.168.0.29  34.209.123.19         6    1518
2  1.521519e+09  192.168.0.29  34.209.123.19         6    1514
3  1.521519e+09  192.168.0.29  34.209.123.19         6    1518
4  1.521519e+09  192.168.0.29  34.209.123.19         6     669
(98255, 5)


In [2]:
import pandas as pd

mapping_path = "device_mapping.csv"

device_map_df = pd.read_csv(
    mapping_path,
    header=None,
    names=["device", "ip"]
)

print(device_map_df.head())
print(device_map_df.columns)

                  device           ip
0                Gateway  192.168.0.1
1            GoogleOnHub  192.168.0.2
2  SamsungSmartThingsHub  192.168.0.4
3          PhilipsHUEHub  192.168.0.5
4             InsteonHub  192.168.0.6
Index(['device', 'ip'], dtype='object')


In [3]:
# CHANGE these based on your actual CSV column names
DEVICE_COL = "device"
IP_COL = "ip"

device_map_df = device_map_df[[DEVICE_COL, IP_COL]].dropna()

# Convert to dictionary:
# "192.168.1.10" -> "Amazon Echo"
ip_to_device = dict(zip(device_map_df[IP_COL].astype(str), device_map_df[DEVICE_COL].astype(str)))

print(ip_to_device)

{'192.168.0.1': 'Gateway', '192.168.0.2': 'GoogleOnHub', '192.168.0.4': 'SamsungSmartThingsHub', '192.168.0.5': 'PhilipsHUEHub', '192.168.0.6': 'InsteonHub', '192.168.0.7': 'Sonos', '192.168.0.8': 'SecurifiAlmond', '192.168.0.10': 'NestCamera', '192.168.0.12': 'BelkinWeMoMotionSensor', '192.168.0.13': 'LIFXVirtualBulb', '192.168.0.14': 'BelkinWeMoSwitch', '192.168.0.15': 'AmazonEchoGen1', '192.168.0.16': 'WinkHub', '192.168.0.17': 'NestProtect', '192.168.0.18': 'BelkinNetcam', '192.168.0.19': 'RingDoorbell', '192.168.0.21': 'RokuTV', '192.168.0.22': 'Roku4', '192.168.0.23': 'AmazonFireTV', '192.168.0.24': 'nVidiaShield', '192.168.0.25': 'AppleTV(4thGen)', '192.168.0.26': 'BelkinWeMoLink', '192.168.0.27': 'NetgearArloCamera', '192.168.0.28': 'D-LinkDCS-5009LCamera', '192.168.0.29': 'LogitechLogiCircle', '192.168.0.30': 'Canary', '192.168.0.31': 'PiperNV', '192.168.0.32': 'WithingsHome', '192.168.0.33': 'BelkinWeMoCrockpot', '192.168.0.34': 'MiCasaVerdeVeraLite', '192.168.0.35': 'Chinese

In [4]:
# -----------------------------
# 2. Add device + direction to each packet
# -----------------------------

def find_device_and_direction(row):
    src = str(row["src_ip"])
    dst = str(row["dst_ip"])

    if src in ip_to_device:
        return pd.Series([ip_to_device[src], "outgoing"])
    elif dst in ip_to_device:
        return pd.Series([ip_to_device[dst], "incoming"])
    else:
        return pd.Series([None, None])

df[["device", "direction"]] = df.apply(find_device_and_direction, axis=1)

# Keep only packets related to known IoT devices
df_device = df.dropna(subset=["device"]).copy()

print(df_device.head())
print(df_device.shape)

           time        src_ip         dst_ip  protocol  length  \
0  1.521519e+09  192.168.0.29  34.209.123.19         6    1514   
1  1.521519e+09  192.168.0.29  34.209.123.19         6    1518   
2  1.521519e+09  192.168.0.29  34.209.123.19         6    1514   
3  1.521519e+09  192.168.0.29  34.209.123.19         6    1518   
4  1.521519e+09  192.168.0.29  34.209.123.19         6     669   

               device direction  
0  LogitechLogiCircle  outgoing  
1  LogitechLogiCircle  outgoing  
2  LogitechLogiCircle  outgoing  
3  LogitechLogiCircle  outgoing  
4  LogitechLogiCircle  outgoing  
(62050, 7)


In [5]:
# -----------------------------
# 3. Create time windows
# -----------------------------

window_size = 5  # seconds

df_device["time"] = df_device["time"].astype(float)

# Make time relative to the first packet
df_device["relative_time"] = df_device["time"] - df_device["time"].min()

# Window start: 0, 5, 10, 15, ...
df_device["window_start"] = (
    df_device["relative_time"] // window_size
) * window_size

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# -----------------------------
# 1. Build feature_df first
# -----------------------------

feature_df = df_device.groupby(["window_start", "device"]).agg(
    pkt_count=("length", "count"),
    total_bytes=("length", "sum"),
    outgoing_pkts=("direction", lambda x: (x == "outgoing").sum()),
    incoming_pkts=("direction", lambda x: (x == "incoming").sum()),
).reset_index()

# label = device for device classification
feature_df["label"] = feature_df["device"]

print(feature_df.head())
print("Original feature_df shape:", feature_df.shape)

# -----------------------------
# 2. Check how many samples each device has
# -----------------------------

print(feature_df["label"].value_counts())

   window_start                      device  pkt_count  total_bytes  \
0           0.0              AmazonEchoGen1          4          792   
1           0.0      BelkinWeMoMotionSensor          8          928   
2           0.0  ChamberlainmyQGarageOpener          4          248   
3           0.0       D-LinkDCS-5009LCamera          8         2586   
4           0.0                     Gateway          6         1634   

   outgoing_pkts  incoming_pkts                       label  
0              4              0              AmazonEchoGen1  
1              4              4      BelkinWeMoMotionSensor  
2              2              2  ChamberlainmyQGarageOpener  
3              8              0       D-LinkDCS-5009LCamera  
4              6              0                     Gateway  
Original feature_df shape: (837, 7)
label
LogitechLogiCircle            60
PhilipsHUEHub                 60
SamsungSmartTV                60
SamsungSmartThingsHub         60
Gateway                    